**Week 4 Learning Unkown Summary Stats** \
Date: 7.29 \
Objectives:
- use embedding network defined from last week to learn summary stats of more complex distributions with unknown summaries.
    - Gaussian Mixture
    - Lotka-Volterra

In [ ]:
# Gaussian Mixture
import torch
from sbi import utils as utils
from sbi.inference import simulate_for_sbi
from sbi.utils.user_input_checks import prepare_for_sbi
from sbi.inference import SNPE
import matplotlib.pyplot as plt
import numpy as np
from sbi.neural_nets import posterior_nn
import torch.nn as nn
import torch.nn.functional as F
torch.random.manual_seed(13)

dim = 5 # parameter dim
n_obs = 1000
prior = utils.BoxUniform(low=torch.zeros(dim), high=torch.ones(dim)*3)
# x ~ pi*N(mu1, sd1 ** 2) + (1-pi)*N(mu2,sd**2)
# abitary parameter choices
# theta = [pi, mu1, mu2, sd1, sd2]

# simulator 
def simulator(theta):
    pi = theta[:, 0]
    mu1 = theta[:, 1]
    sd1 = theta[:, 2]
    mu2 = theta[:, 3]
    sd2 = theta[:, 4]
    
    n_batch = theta.shape[0]
    
    x1 = torch.normal(mu1.view(-1, 1), sd1.view(-1, 1).clamp(min=1e-2)).to(theta)
    x2 = torch.normal(mu2.view(-1, 1), sd2.view(-1, 1).clamp(min=1e-2)).to(theta)
    
    pi = pi.view(-1, 1)
    x = pi * x1 + (1 - pi) * x2
    return x.repeat(1, n_obs)

# 3 layer embedding network
class embedding_network(nn.Module):
    def __init__(self, input_dim, embedding_dim):
        super().__init__()
        self.network = nn.Sequential(    #sets up a sequential learning network with Linear + ReLU + Linear layers
            nn.Linear(input_dim, 64),   # input dimension --> 64D
            nn.ReLU(),
            nn.Linear(64, embedding_dim)   # 64D --> output dimension
        )

    def forward(self, x):
        return self.network(x)

# training sbi, regular training method
def train_sbi(net, num_simulations, prior):
    sim, pri = prepare_for_sbi(simulator, prior)
    
    inference = SNPE(prior=pri, density_estimator=posterior_nn(model='maf', embedding_net=net))
    theta, x = simulate_for_sbi(sim, prior, num_simulations=num_simulations)
    x = x.view(x.shape[0], -1)
    
    density_estimator = inference.append_simulations(theta, x).train()
    posterior = inference.build_posterior(density_estimator)
    print('theta:',theta)
    return posterior, net, theta, x
    
# analysis, extract learned summary stats
class analyze_embeddings:
    def __init__(self, embedding_net, x_eval):
        self.embedding_net = embedding_net
        self.eval_x = x_eval
        self.analyze()

    def analyze(self):
        learned_summary = self.embedding_net(self.eval_x)
        self.learned_summary = learned_summary

    # # project learned summary to 1D, compare the first column component only
    #     projected_learned = learned_summary[:, 0:1]

def experiment():
    embedding_dims = [1, 2, 4, 8]
    results = {}

    for dim_embed in embedding_dims:
        print(f"\n Training with embedding_dim = {dim_embed}")
        embed_net = embedding_network(input_dim=n_obs, embedding_dim=dim_embed)
        posterior, net, theta, x = train_sbi(embed_net, num_simulations=n_obs, prior=prior)
        analysis = analyze_embeddings(net, x[:100])
        #use the first 100 samples in x for a quick analysis, adjust as needed for later parts
        results[dim_embed] = {
            # "cosine_similarity": analysis.cosine_similarity,
            # "correlation": analysis.correlation
            # might replace with other metrics
            'learned summaries': analysis.learned_summary
        }

    return results

experiment()


 Training with embedding_dim = 1


C:\Users\boat\AppData\Local\Temp\ipykernel_7508\436315863.py:53: DeprecationWarning: This method is deprecated as of sbi version v0.23.0. and will be removed in a         future release.Please use `process_prior` and `process_simulator` in the future.
  sim, pri = prepare_for_sbi(simulator, prior)


  0%|          | 0/10000 [00:00<?, ?it/s]

 Neural network successfully converged after 325 epochs.theta: tensor([[1.2044e+00, 1.1503e-01, 2.2974e+00, 2.4689e+00, 1.0182e-03],
        [7.1394e-01, 2.0943e-01, 2.8689e-02, 7.8764e-01, 5.9844e-01],
        [5.2469e-02, 2.8135e+00, 1.2487e+00, 1.1332e+00, 1.2566e+00],
        ...,
        [2.2033e-01, 1.2637e+00, 2.9920e+00, 2.3412e+00, 2.1665e+00],
        [1.1410e+00, 5.5383e-01, 7.2248e-01, 1.3598e+00, 2.1894e+00],
        [1.2486e+00, 1.3258e+00, 1.9261e+00, 1.3322e-01, 1.4673e+00]])

 Training with embedding_dim = 2


  0%|          | 0/10000 [00:00<?, ?it/s]

...

  0%|          | 0/10000 [00:00<?, ?it/s]

 Neural network successfully converged after 312 epochs.theta: tensor([[0.1214, 2.9425, 0.9311, 2.0258, 1.4223],
        [2.5065, 0.8422, 0.0828, 1.5262, 2.2317],
        [1.1607, 0.0314, 0.4594, 1.9130, 2.0303],
        ...,
        [0.0377, 0.2441, 0.8838, 1.7688, 0.3426],
        [0.8740, 0.4700, 2.8570, 0.1127, 2.4687],
        [1.4968, 1.2822, 1.7508, 1.2260, 1.5399]])

 Training with embedding_dim = 8


  0%|          | 0/10000 [00:00<?, ?it/s]

 Neural network successfully converged after 211 epochs.theta: tensor([[2.3283, 1.3914, 2.7241, 1.5709, 2.8247],
        [2.6191, 1.5505, 1.5172, 0.3440, 2.2855],
        [1.9006, 1.3913, 2.3485, 1.4184, 1.4479],
        ...,
        [1.9269, 1.3155, 0.2602, 0.9104, 0.2803],
        [2.5435, 0.0758, 0.1159, 2.5331, 1.3485],
        [1.2236, 2.8984, 1.8815, 1.3515, 2.7242]])


{1: {'learned summaries': tensor([[-188.1571],
          [ -21.4259],
          [ -14.5040],
          [ -69.7253],
          [-356.7585],
          [ -42.3539],
          [  -6.5439],
          [ -35.6166],
          [ -83.1125],
          [-277.1865],
          [ -15.4195],
          [-287.0281],
          [-123.5536],
          [-156.6947],
          [-396.1457],
          [ -18.8568],
          [  -2.4402],
          [-183.4870],
          [ -16.0245],
          [ -77.3032],
          [ -61.1311],
          [ -10.9672],
          [ -56.2256],
          [ -48.3804],
          [ -63.0765],
          [-120.8845],
          [-102.4750],
          [-364.6045],
          [ -12.4871],
          [ -62.1158],
          [ -83.0348],
          [ -95.3834],
          [-167.8126],
          [ -77.9092],
          [-182.9843],
          [ -78.5853],
          [-109.8890],
          [-279.4376],
          [ -75.6092],
          [  -8.0426],
          [ -32.3996],
          [-160.4961],
          